# Entity-Annotated Dataset: Dictionary and Model, with Timelines and Co-occurrence

This notebook builds a per-article biomedical entity dataset two ways and derives timelines and co-occurrence from each:

- Part A, dictionary (CPU, fast): regex matching of a curated, model-extended term list against every abstract. Runs in minutes, deterministic, always available.
- Part B, scispaCy model (GPU, heavy): the trained `en_ner_bc5cdr_md` model run over the abstracts, saving entities per article. Higher recall, catches rare and unlisted entities, but takes hours. Controlled by a switch so it runs only when asked, and caches so it runs once.
- Part C: both feed the same downstream code (timelines, co-occurrence), so the two methods are directly comparable.

The model pass here saves entities per article, which is what makes a model-based version of the timelines and co-occurrence possible. Both methods write their per-article table and aggregates to `data/3_entities/`.

Runs on the local clean corpus (needs abstract text). The model pass additionally needs scispaCy, the model, and (for speed) a working CuPy/GPU setup.

## How the two methods relate

The dictionary is fast because it is pure text matching on the CPU; a GPU cannot accelerate regular expressions. The model is slow because it is a neural network; a GPU accelerates its maths greatly, which is why Part B includes GPU setup. The dictionary's term list was extended with the model's most frequent discoveries, so even the fast method carries much of the model's knowledge. Run Part A always; run Part B when higher recall is worth the time, then rely on its cache.

## Setup

In [ ]:
import os, glob, collections, itertools, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

sns.set_theme(style="whitegrid")

ROOT = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(ROOT, "data", "2_clean")
OUT_DIR = os.path.join(ROOT, "data", "3_entities")
CACHE_DIR = os.path.join(ROOT, "data", "cache")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

import pyarrow.parquet as pq
_avail = set()
for f in glob.glob(os.path.join(DATA_DIR, "*.parquet")):
    _avail |= set(pq.ParquetFile(f).schema.names); break
use_cols = [c for c in ["uid", "year", "abstract", "n_authors", "journal"] if c in _avail] or ["uid", "year", "abstract"]

df = pd.read_parquet(DATA_DIR, columns=use_cols)
df = df[df["year"] <= 2025].copy()
if "n_authors" not in df: df["n_authors"] = np.nan
if "journal" not in df: df["journal"] = ""
HAS_ABSTRACTS = bool((df["abstract"].fillna("").str.len() > 0).any())
print(f"loaded {len(df):,} records | columns: {list(df.columns)} | abstracts: {HAS_ABSTRACTS}")

## Shared helper: build timelines and co-occurrence from any entity columns

Both methods produce a disease-list column and a chemical-list column per article. This function turns those into the four aggregate tables, so the same code serves dictionary and model results.

In [ ]:
# Noise filter for model entities: abbreviations, symbols, and self-pairs.
ENTITY_NOISE = {
    "±", "+", "-", "/", "=", "~", "n", "p", "i", "ii", "iii", "iv", "a", "b", "c", "d",
    "ad", "hf", "mi", "ckd", "copd", "ra", "ms", "pd", "tb", "ht", "dm",   # bare abbreviations
    "vs", "ci", "or", "rr", "hr", "sd", "se", "iqr",                        # stats notation
}
def is_clean(term):
    """Reject noise: symbols, very short tokens, pure numbers, known abbreviations."""
    t = term.strip().lower()
    if t in ENTITY_NOISE: return False
    if len(t) <= 2: return False            # 1-2 char tokens are almost always noise
    if any(ch.isdigit() for ch in t) and not any(ch.isalpha() for ch in t): return False
    if not any(ch.isalpha() for ch in t): return False   # must contain a letter
    return True

def clean_entities(lst):
    return [t for t in lst if is_clean(t)]


def build_and_save_aggregates(data, dz_col, chem_col, tag, filter_noise=False):
    """Build and save timelines, co-occurrence, AND single-entity frequency tables.
    If filter_noise, drop abbreviation/symbol noise (use for model; dictionary is already clean)."""
    if filter_noise:
        data = data.copy()
        data[dz_col] = data[dz_col].map(clean_entities)
        data[chem_col] = data[chem_col].map(clean_entities)

    articles_per_year = data.groupby("year").size()

    # SINGLE-ENTITY frequency tables (the "top entities" ranking) - NEW
    dz_freq = collections.Counter(t for lst in data[dz_col] for t in set(lst))
    chem_freq = collections.Counter(t for lst in data[chem_col] for t in set(lst))
    dz_freq_df = (pd.DataFrame(dz_freq.items(), columns=["disease", "n_articles"])
                  .sort_values("n_articles", ascending=False).reset_index(drop=True))
    chem_freq_df = (pd.DataFrame(chem_freq.items(), columns=["chemical", "n_articles"])
                    .sort_values("n_articles", ascending=False).reset_index(drop=True))
    dz_freq_df.to_parquet(os.path.join(OUT_DIR, f"disease_frequency_{tag}.parquet"))
    chem_freq_df.to_parquet(os.path.join(OUT_DIR, f"chemical_frequency_{tag}.parquet"))

    # timelines
    def timeline_table(col):
        rec = collections.defaultdict(collections.Counter)
        for yr, lst in zip(data["year"], data[col]):
            for term in set(lst):
                rec[term][yr] += 1
        return pd.DataFrame(rec).fillna(0).astype(int).reindex(sorted(set(data["year"])))
    dz_tl = timeline_table(dz_col); chem_tl = timeline_table(chem_col)
    dz_tl.to_parquet(os.path.join(OUT_DIR, f"disease_timeline_{tag}.parquet"))
    chem_tl.to_parquet(os.path.join(OUT_DIR, f"chemical_timeline_{tag}.parquet"))

    # co-occurrence (skip self-pairs where disease == chemical string)
    dc, dd = collections.Counter(), collections.Counter()
    for dz_list, ch_list in zip(data[dz_col], data[chem_col]):
        ds = set(dz_list)
        for a in ds:
            for b in set(ch_list):
                if a != b:                      # skip self-pairs (ad+ad, hf+hf)
                    dc[(a, b)] += 1
        dd.update(p for p in itertools.combinations(sorted(ds), 2) if p[0] != p[1])
    dc_df = (pd.DataFrame([(a, b, c) for (a, b), c in dc.items()], columns=["disease", "chemical", "n_articles"])
             .sort_values("n_articles", ascending=False).reset_index(drop=True))
    dd_df = (pd.DataFrame([(a, b, c) for (a, b), c in dd.items()], columns=["disease_a", "disease_b", "n_articles"])
             .sort_values("n_articles", ascending=False).reset_index(drop=True))
    dc_df.to_parquet(os.path.join(OUT_DIR, f"disease_chemical_pairs_{tag}.parquet"))
    dd_df.to_parquet(os.path.join(OUT_DIR, f"disease_disease_pairs_{tag}.parquet"))

    print(f"[{tag}] saved: disease/chemical frequency, 2 timelines, 2 co-occurrence tables")
    print(f"[{tag}] top diseases: {', '.join(dz_freq_df['disease'].head(8))}")
    return dz_tl, chem_tl, dc_df, dd_df, articles_per_year


def plot_top_timeline(timeline, articles_per_year, freq_source_col, data, title):
    freq = collections.Counter(t for lst in data[freq_source_col] for t in lst)
    top = [t for t, _ in freq.most_common(8) if t in timeline.columns]
    plt.figure(figsize=(13, 6))
    for term in top:
        s = timeline[term].reindex(articles_per_year.index, fill_value=0) / articles_per_year * 1000
        plt.plot(s.index, s.values, marker="o", markersize=3, label=term)
    plt.title(title); plt.xlabel("year"); plt.ylabel("per 1,000 articles")
    plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8, frameon=False)
    plt.tight_layout(); plt.show()

# Part A: Dictionary method (CPU, always runs)

## A1. Term lists (model-extended, curated)
Loaded from notebook 08's saved file if present, else curated seeds. A DROP set removes obvious model mislabels (symptoms as diseases, behaviours as chemicals).

In [ ]:
SEED_DISEASE = {
    "covid-19", "covid", "sars-cov-2", "diabetes", "hypertension", "asthma", "cancer",
    "breast cancer", "prostate cancer", "lung cancer", "stroke", "obesity", "influenza",
    "pneumonia", "tuberculosis", "hiv", "sepsis", "depression", "anxiety", "alzheimer",
    "alzheimer's disease", "parkinson", "arthritis", "epilepsy", "leukemia", "melanoma",
    "dementia", "schizophrenia", "ptsd", "adhd", "fibrosis", "inflammation", "infection",
    "heart failure", "myocardial infarction", "atrial fibrillation", "cardiovascular disease",
}
SEED_CHEMICAL = {
    "aspirin", "metformin", "insulin", "ibuprofen", "remdesivir", "warfarin", "statin",
    "heparin", "morphine", "paracetamol", "acetaminophen", "penicillin", "dexamethasone",
    "prednisone", "methotrexate", "tamoxifen", "lithium", "ketamine", "fluoxetine",
    "glucose", "cholesterol", "estrogen", "dopamine", "cortisol", "testosterone",
    "creatinine", "nicotine", "vitamin d", "cisplatin", "nitric oxide",
}
DROP = {
    "death", "deaths", "pain", "trauma", "bleeding", "fatigue", "toxicity", "psychiatric",
    "tumors", "cancers", "infections", "fractures",
    "smoking", "alcohol", "oxygen", "calcium", "sodium", "iron", "atp", "snp",
    "amino acid", "nucleotide", "steroid",
}

terms_path = os.path.join(ROOT, "data", "config", "entity_terms_extended.json")
if os.path.exists(terms_path):
    with open(terms_path) as f:
        _t = json.load(f)
    DISEASE_TERMS = set(_t["diseases"]) - DROP
    CHEMICAL_TERMS = set(_t["chemicals"]) - DROP
    print(f"loaded model-extended terms (curated): {len(DISEASE_TERMS)} diseases, {len(CHEMICAL_TERMS)} chemicals")
else:
    DISEASE_TERMS = SEED_DISEASE - DROP
    CHEMICAL_TERMS = SEED_CHEMICAL - DROP
    print(f"extended file not found; curated seeds: {len(DISEASE_TERMS)} diseases, {len(CHEMICAL_TERMS)} chemicals")

def build_matcher(vocab):
    return re.compile(r"\b(" + "|".join(re.escape(t) for t in sorted(vocab, key=len, reverse=True)) + r")\b")

DISEASE_RE = build_matcher(DISEASE_TERMS)
CHEMICAL_RE = build_matcher(CHEMICAL_TERMS)

def extract(text, matcher):
    if not isinstance(text, str):
        return []
    return list(set(matcher.findall(text.lower())))

## A2. Extract, build per-article table, save

In [ ]:
if HAS_ABSTRACTS:
    tqdm.pandas(desc="dict diseases")
    df["dz_dict"] = df["abstract"].progress_map(lambda t: extract(t, DISEASE_RE))
    tqdm.pandas(desc="dict chemicals")
    df["chem_dict"] = df["abstract"].progress_map(lambda t: extract(t, CHEMICAL_RE))

    dict_df = df[["uid", "year", "n_authors", "journal", "dz_dict", "chem_dict"]].copy()
    dict_df["n_diseases"] = dict_df["dz_dict"].map(len)
    dict_df["n_chemicals"] = dict_df["chem_dict"].map(len)
    dict_df.to_parquet(os.path.join(OUT_DIR, "article_entities_dictionary.parquet"))
    print(f"saved article_entities_dictionary.parquet ({len(dict_df):,} rows)")
    print(f"  with >=1 disease: {(dict_df['n_diseases']>0).mean()*100:.0f}%  "
          f"chemical: {(dict_df['n_chemicals']>0).mean()*100:.0f}%")

## A3. Dictionary timelines and co-occurrence

In [ ]:
if HAS_ABSTRACTS:
    dz_tl_d, chem_tl_d, dc_d, dd_d, apy = build_and_save_aggregates(df, "dz_dict", "chem_dict", "dictionary")
    plot_top_timeline(dz_tl_d, apy, "dz_dict", df, "Dictionary: top disease prevalence (per 1,000 articles)")
    print("\ntop disease-chemical (dictionary):")
    for _, r in dc_d.head(12).iterrows():
        print(f"  {r['n_articles']:>7,}   {r['disease']}  +  {r['chemical']}")

**What this shows:** the dictionary view, fast and full-corpus, on the curated model-extended vocabulary. These are the saved `*_dictionary.parquet` tables. The model pass below produces the same shape with higher recall.

# Part B: scispaCy model method (GPU, runs on demand)

This pass runs the neural NER model and saves entities PER ARTICLE (which notebook 08 did not), enabling a model-based version of the timelines and co-occurrence. It is gated by RUN_MODEL so it does not start a multi-hour job unless asked, and it caches so it runs once.

## B1. GPU setup
Registers the pip-installed CUDA DLL directories (Windows) and confirms CuPy works, before importing spaCy. Skips gracefully if anything is missing.

In [ ]:
# Set this True to run the model pass. False = skip (dictionary results still complete).
RUN_MODEL = True
# SAMPLE_N: an int runs a sample; None runs the full corpus (hours). Sample first to validate.
SAMPLE_N = None

gpu_ok = False
if RUN_MODEL:
    # 1. make the pip-installed CUDA DLLs findable (Windows; harmless elsewhere)
    try:
        nvidia_root = os.path.join(__import__("sys").prefix, "Lib", "site-packages", "nvidia")
        for bindir in glob.glob(os.path.join(nvidia_root, "*", "bin")):
            if os.path.isdir(bindir):
                os.add_dll_directory(bindir)
                os.environ["PATH"] = bindir + os.pathsep + os.environ.get("PATH", "")
    except Exception as e:
        print(f"(DLL path setup skipped: {type(e).__name__})")

    # 2. confirm CuPy works (proves the GPU stack is functional)
    try:
        import cupy as cp
        _ = cp.array([1, 2, 3]).sum()
        gpu_ok = True
        print(f"CuPy works. GPU devices: {cp.cuda.runtime.getDeviceCount()}")
    except Exception as e:
        print(f"CuPy not working ({type(e).__name__}); model will run on CPU if it runs at all.")
else:
    print("RUN_MODEL is False; skipping the model pass. Dictionary results (Part A) are complete.")

## B2. Load the model on GPU and verify it extracts entities
Keeps tok2vec (NER depends on it); disables only the independent components. The single-sentence test must return real entities before any large run.

In [ ]:
nlp = None
if RUN_MODEL:
    try:
        import scispacy
        import spacy
        if gpu_ok:
            spacy.require_gpu()
            print("spaCy GPU enabled.")
        # KEEP tok2vec (NER needs it). Disable only independent components.
        nlp = spacy.load("en_ner_bc5cdr_md", disable=["tagger", "parser", "attribute_ruler", "lemmatizer"])
        print("model loaded. pipeline:", nlp.pipe_names)

        _doc = nlp("The patient was treated with aspirin for diabetes and hypertension.")
        _ents = [(e.text, e.label_) for e in _doc.ents]
        print("verification entities:", _ents)
        assert any(lbl == "DISEASE" for _, lbl in _ents), "NER returned no diseases; check tok2vec is present"
        print("verification passed: model extracts entities correctly.")
    except Exception as e:
        print(f"model unavailable ({type(e).__name__}): {e}")
        nlp = None

## B3. Run the model per article, save per-article entities, build aggregates
Caches the per-article model table; on re-open it loads instead of re-running. Truncates abstracts to 1500 chars for speed.

In [ ]:
if RUN_MODEL and HAS_ABSTRACTS and nlp is not None:
    tag = "model_full" if SAMPLE_N is None else f"model_{min(SAMPLE_N, len(df))}"
    model_table_path = os.path.join(OUT_DIR, f"article_entities_{tag}.parquet")

    if os.path.exists(model_table_path):
        model_df = pd.read_parquet(model_table_path)
        print(f"loaded cached model per-article table: {model_table_path} ({len(model_df):,} rows)")
    else:
        if SAMPLE_N is None:
            sub = df
            print(f"MODEL: FULL CORPUS ({len(df):,} abstracts). This can take HOURS.")
        else:
            sub = df.sample(min(SAMPLE_N, len(df)), random_state=42)
            print(f"MODEL: sample of {len(sub):,} abstracts.")

        uids = sub["uid"].tolist()
        years = sub["year"].tolist()
        texts = sub["abstract"].fillna("").str[:1500].tolist()

        rec_dz, rec_chem = [], []
        for doc in tqdm(nlp.pipe(texts, batch_size=512, n_process=1), total=len(texts), desc="scispaCy NER"):
            d_set, c_set = set(), set()
            for e in doc.ents:
                lab = e.label_.upper(); term = e.text.lower().strip()
                if lab == "DISEASE": d_set.add(term)
                elif lab == "CHEMICAL": c_set.add(term)
            rec_dz.append(sorted(d_set)); rec_chem.append(sorted(c_set))

        model_df = pd.DataFrame({"uid": uids, "year": years, "dz_model": rec_dz, "chem_model": rec_chem})
        model_df["n_diseases"] = model_df["dz_model"].map(len)
        model_df["n_chemicals"] = model_df["chem_model"].map(len)
        model_df.to_parquet(model_table_path)
        print(f"saved model per-article table to {model_table_path}")

    # aggregates from the model columns (same helper as the dictionary)
    dz_tl_m, chem_tl_m, dc_m, dd_m, apy_m = build_and_save_aggregates(model_df, "dz_model", "chem_model", tag, filter_noise=True)
    plot_top_timeline(dz_tl_m, apy_m, "dz_model", model_df, f"Model: top disease prevalence (per 1,000 articles) [{tag}]")
    print("\ntop disease-chemical (model):")
    for _, r in dc_m.head(12).iterrows():
        print(f"  {r['n_articles']:>7,}   {r['disease']}  +  {r['chemical']}")
else:
    print("model pass not run (RUN_MODEL False, no abstracts, or model unavailable).")

In [ ]:
# Inspect the FULL saved tables (not just the printed top rows).
import pandas as pd, os, glob
ENT = os.path.join(ROOT, "data", "3_entities")

# Auto-detect which method tags were actually produced (dictionary always; model if it ran).
_freq_files = glob.glob(os.path.join(ENT, "disease_frequency_*.parquet"))
_tags = sorted(os.path.basename(f).replace("disease_frequency_", "").replace(".parquet", "") for f in _freq_files)
print("available tags:", _tags)

# Inspect each available tag.
for TAG in _tags:
    print(f"\n===== {TAG} =====")
    dz_freq = pd.read_parquet(os.path.join(ENT, f"disease_frequency_{TAG}.parquet"))
    chem_freq = pd.read_parquet(os.path.join(ENT, f"chemical_frequency_{TAG}.parquet"))
    dc_path = os.path.join(ENT, f"disease_chemical_pairs_{TAG}.parquet")
    dc = pd.read_parquet(dc_path) if os.path.exists(dc_path) else pd.DataFrame()

    print(f"  distinct diseases:  {len(dz_freq):,}")
    print(f"  distinct chemicals: {len(chem_freq):,}")
    print(f"  disease-chemical pairs: {len(dc):,}")
    print("\n  top 20 diseases:")
    print(dz_freq.head(20).to_string(index=False))
    print("\n  top 20 disease-chemical pairs:")
    if len(dc):
        print(dc.head(20).to_string(index=False))

**What this shows:** when run, the model produces the same per-article and aggregate tables as the dictionary, but with higher recall, including entities and variants the dictionary lacks. Because counts and tables share the dictionary's structure, the two are directly comparable side by side.

# Part C: Summary and caveats

### What this notebook produced and saved (in `data/3_entities/`)
Dictionary method (always): `article_entities_dictionary.parquet`, plus `disease_timeline_dictionary`, `chemical_timeline_dictionary`, `disease_chemical_pairs_dictionary`, `disease_disease_pairs_dictionary`. Model method (when RUN_MODEL is True): the same set tagged `model_<n>` or `model_full`, including the per-article model entities that notebook 08 did not save. Both sets share a structure, so downstream work loads whichever it wants.

### How to run the model pass
Set RUN_MODEL = True. Leave SAMPLE_N at 50000 for a fast validation run first; confirm the verification entities print and the sample output looks right. Then set SAMPLE_N = None for the full corpus and run once; the per-article table caches, so reopening loads it instead of recomputing.

### Caveats
Dictionary recall is a lower bound, even extended and curated. The model has higher recall but imprecise category edges (it tags some symptoms as diseases), so its raw entities carry more noise. Co-occurrence is association from co-mention, not causation. Counts are document-level (one per article per term). Both methods see abstracts only, within the filtered corpus scope. The model's GPU speed depends on a working CuPy/CUDA setup; without it the model falls back to CPU and the full corpus becomes impractically slow.

---

## Notebook complete

This notebook built the entity dataset both ways: a fast dictionary pass (always) and an on-demand scispaCy model pass (GPU, saving per-article entities), with shared timelines and co-occurrence so the two are comparable. Downstream notebooks can load either set from `data/3_entities/`.